# Condition classifier: 6 oral conditions from a phone photo

**Data:** the Roboflow *oral-diseases* v2 export (6 conditions, 9,155 photos), prepared by
`models/classifier/scripts/Aiden/prepare_roboflow_classifier.py`. It has **no Normal (healthy) class**, so every photo
is assigned one of the 6 conditions.
Roboflow's training split holds 3 augmented copies of every photo; validation and test are un-augmented.
**Model:** DenseNet-121 pretrained on ImageNet, fine-tuned in two stages.
**Main metric:** macro-F1 and per-class recall, because the classes are imbalanced and a missed problem (false negative) is the costly error.

Runs locally from the repo (data in `data/processed/classifier_roboflow`, outputs to `models/classifier/results/Aiden`),
or on Kaggle with **GPU T4** and **Internet on** (for the pretrained weights): upload the `classifier_roboflow` folder
as a private Kaggle dataset and add it with *Add Input*.

## 1. Setup

In [ ]:
import os, random
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score, accuracy_score

SEED, IMG, BATCH = 42, 224, 32
FROZEN_EPOCHS, FINETUNE_EPOCHS, PATIENCE = 5, 25, 5
DUP_SIM = 0.95          # cosine similarity above which two images count as the same photo (check the examples it prints)

SMOKE = os.environ.get("SMOKE") == "1"   # tiny CPU test run only; leave unset for real runs
ON_KAGGLE = Path("/kaggle/input").exists()
if SMOKE:
    DATA_DIR, OUT_DIR = Path(os.environ["DATA_DIR"]), Path(os.environ["OUT_DIR"])
    FROZEN_EPOCHS, FINETUNE_EPOCHS, DUP_SIM = 1, 1, 0.9999
elif ON_KAGGLE:
    DATA_DIR = next(Path("/kaggle/input").rglob("manifest.csv")).parent
    OUT_DIR = Path("/kaggle/working")
else:
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
    DATA_DIR = REPO / "data/processed/classifier_roboflow"
    OUT_DIR = REPO / "models/classifier/results/Aiden"
    OUT_DIR.mkdir(parents=True, exist_ok=True)
assert (DATA_DIR / "manifest.csv").exists(), f"no manifest.csv in {DATA_DIR}: run prepare_roboflow_classifier.py first"
PRETRAINED = not SMOKE

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("data:", DATA_DIR, "| device:", DEV)

## 2. Look at the data

In [ ]:
df = pd.read_csv(DATA_DIR / "manifest.csv")
CLASSES = sorted(df.label.unique())
counts = pd.crosstab(df.label, df.split)[["train", "val", "test"]]
print(counts, f"\ntotal {len(df)} images from {df.group.nunique()} photos")
counts.plot.barh(stacked=True, title="Images per class (train includes Roboflow's 3 copies per photo)"); plt.show()

## 3. Catch near-duplicates across splits

Every augmented copy of a photo already shares its `group`. But the same scene can also appear as two different photos
(re-uploaded, cropped, or the same mouth shot twice). Here one image per photo is turned into a 1024-number description by
the pretrained network; photos whose descriptions are almost identical are merged into one **group**.
Any validation or test photo that ends up in a group with training photos is dropped, so the test set never contains a
copy of a training photo (data leakage).

In [ ]:
weights = models.DenseNet121_Weights.IMAGENET1K_V1 if PRETRAINED else None
NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
eval_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(), NORM])

backbone = models.densenet121(weights=weights).to(DEV).eval()
first = df.drop_duplicates("group")   # one image per photo: its augmented copies are already grouped
embs = []
with torch.no_grad():
    for i in range(0, len(first), 64):
        x = torch.stack([eval_tf(Image.open(DATA_DIR / f).convert("RGB")) for f in first.file[i:i + 64]]).to(DEV)
        e = torch.relu(backbone.features(x)).mean((2, 3))   # DenseNet's own pooling step
        embs.append(nn.functional.normalize(e, dim=1).cpu())
E = torch.cat(embs)
sim = (E @ E.T).numpy()
np.fill_diagonal(sim, 0)
_, comp = connected_components(csr_matrix(sim > DUP_SIM), directed=False)
df["photo"] = df.group
df["group"] = df.photo.map(dict(zip(first.group, comp)))
n_photos = df.groupby("group").photo.nunique()
print(f"{(n_photos > 1).sum()} near-duplicate groups covering {n_photos[n_photos > 1].sum()} photos")

mixed = df.groupby("group").label.nunique()
mixed = mixed[mixed > 1].index
print(f"dropping {df.group.isin(mixed).sum()} images whose near-duplicates carry a different label")
df = df[~df.group.isin(mixed)]
leak = (df.split != "train") & df.group.isin(df[df.split == "train"].group)
print(f"dropping {leak.sum()} val/test images that are near-copies of a training photo")
df = df[~leak].reset_index(drop=True)

In [ ]:
# Show a few groups so you can judge whether DUP_SIM is right (too low = different photos merged)
n_photos = df.groupby("group").photo.nunique()
shown = df[df.group.isin(n_photos[n_photos > 1].index)].drop_duplicates("photo").groupby("group").head(3).groupby("group")
for k, (g, rows) in enumerate(list(shown)[:4]):
    fig, axes = plt.subplots(1, len(rows), figsize=(3 * len(rows), 3))
    for ax, r in zip(np.atleast_1d(axes), rows.itertuples()):
        ax.imshow(Image.open(DATA_DIR / r.file)); ax.set_title(f"{r.label} ({r.split})", fontsize=8); ax.axis("off")
    plt.show()

## 4. Split: Roboflow's train / valid / test, after the near-duplicate check

In [ ]:
assert not set(df[df.split == "train"].group) & set(df[df.split != "train"].group), "leak: group in train and val/test"
df.to_csv(OUT_DIR / "splits.csv", index=False)
pd.crosstab(df.label, df.split)

## 5. Data loading and augmentation

Augmentation only changes position and light a little. **No hue or saturation changes:** tooth discolouration *is* a colour, so changing colour would teach the model to ignore it.

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG + 16, IMG + 16)), transforms.RandomCrop(IMG),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(), NORM])

class OralDS(Dataset):
    def __init__(self, frame, tf):
        self.frame, self.tf = frame.reset_index(drop=True), tf
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        r = self.frame.iloc[i]
        return self.tf(Image.open(DATA_DIR / r.file).convert("RGB")), CLASSES.index(r.label)

part = {s: df[df.split == s] for s in ("train", "val", "test")}
workers = 2 if ON_KAGGLE else 0   # Windows Jupyter cannot start loader workers for a class defined in the notebook
train_dl = DataLoader(OralDS(part["train"], train_tf), BATCH, shuffle=True, num_workers=workers)
val_dl = DataLoader(OralDS(part["val"], eval_tf), BATCH, num_workers=workers)
test_dl = DataLoader(OralDS(part["test"], eval_tf), BATCH, num_workers=workers)

## 6. Model and class-weighted loss

Rare classes (e.g. Caries) get a larger weight in the loss, so the model is penalised more for missing them.

In [ ]:
def new_model():
    m = models.densenet121(weights=weights)
    m.classifier = nn.Linear(m.classifier.in_features, len(CLASSES))
    return m.to(DEV)

n = part["train"].label.value_counts().reindex(CLASSES).values
class_w = torch.tensor(len(part["train"]) / (len(CLASSES) * n), dtype=torch.float32, device=DEV)
crit = nn.CrossEntropyLoss(weight=class_w)
print(dict(zip(CLASSES, class_w.cpu().numpy().round(2))))

## 7. Training: frozen backbone first, then fine-tune

The best epoch is chosen by **validation** macro-F1. The test set is not touched until section 8.

In [ ]:
def run_epoch(model, loader, opt=None):
    training = opt is not None
    model.train(training)
    total, ys, preds = 0.0, [], []
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEV), y.to(DEV)
            out = model(x)
            loss = crit(out, y)
            if training:
                opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(y)
            ys += y.tolist(); preds += out.argmax(1).tolist()
    return total / len(ys), f1_score(ys, preds, average="macro"), ys, preds

history, best = [], {"f1": -1.0}

def fit(model, epochs, lr, stage):
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    stale = 0
    for ep in range(1, epochs + 1):
        tl, tf1, _, _ = run_epoch(model, train_dl, opt)
        vl, vf1, _, _ = run_epoch(model, val_dl)
        history.append(dict(stage=stage, epoch=ep, train_loss=tl, train_f1=tf1, val_loss=vl, val_f1=vf1))
        print(f"{stage} {ep:2d}  train loss {tl:.3f} f1 {tf1:.3f} | val loss {vl:.3f} f1 {vf1:.3f}")
        if vf1 > best["f1"]:
            best["f1"] = vf1
            torch.save(model.state_dict(), OUT_DIR / "best.pt")
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print("early stop: validation F1 stopped improving"); break

model = new_model()
for p in model.features.parameters():
    p.requires_grad = False
fit(model, FROZEN_EPOCHS, 1e-3, "frozen")
for p in model.parameters():
    p.requires_grad = True
fit(model, FINETUNE_EPOCHS, 1e-4, "finetune")

In [ ]:
h = pd.DataFrame(history)
h.to_csv(OUT_DIR / "history.csv", index=False)
ax = h[["train_loss", "val_loss"]].plot(title="Loss (train well below val = overfitting)")
h[["train_f1", "val_f1"]].plot(title="Macro-F1"); plt.show()

## 8. Final test (run once, at the end)

In [ ]:
model.load_state_dict(torch.load(OUT_DIR / "best.pt", map_location=DEV))
_, test_f1, ys, preds = run_epoch(model, test_dl)
print(f"test accuracy {accuracy_score(ys, preds):.3f} | macro-F1 {test_f1:.3f}\n")
report = classification_report(ys, preds, labels=range(len(CLASSES)), target_names=CLASSES, digits=3, zero_division=0)
print(report)
(OUT_DIR / "test_report.txt").write_text(report)
ConfusionMatrixDisplay.from_predictions(ys, preds, labels=range(len(CLASSES)), display_labels=CLASSES,
                                        xticks_rotation=45, cmap="Blues")
plt.title("Test set confusion matrix"); plt.show()

## 9. Grad-CAM: where is the model looking?

Red areas pushed the prediction most. If the heat sits on lips, background or watermarks instead of teeth, the model learned a shortcut.

In [ ]:
def grad_cam(model, x):
    # DenseNet's forward() is written out here so we can take the gradient at its last feature maps
    # (a hook would break: DenseNet applies an in-place ReLU right after them)
    model.eval()
    act = torch.relu(model.features(x))
    out = model.classifier(act.mean((2, 3)))
    cls = out.argmax(1).item()
    grad, = torch.autograd.grad(out[0, cls], act)
    w = grad.mean((2, 3), keepdim=True)
    cam = torch.relu((w * act).sum(1))[0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.detach().cpu().numpy(), cls, out.softmax(1)[0, cls].item()

sample = part["test"].groupby("label").head(1)
fig, axes = plt.subplots(1, len(sample), figsize=(3.2 * len(sample), 3.4))
for ax, r in zip(np.atleast_1d(axes), sample.itertuples()):
    img = Image.open(DATA_DIR / r.file).convert("RGB").resize((IMG, IMG))
    cam, cls, p = grad_cam(model, eval_tf(img).unsqueeze(0).to(DEV))
    ax.imshow(img); ax.imshow(Image.fromarray(np.uint8(cam * 255)).resize((IMG, IMG)), cmap="jet", alpha=0.4)
    ax.set_title(f"true {r.label}\npred {CLASSES[cls]} {p:.0%}", fontsize=8); ax.axis("off")
plt.show()

## 10. Save the model for the app

In [ ]:
torch.save({"state_dict": model.state_dict(), "classes": CLASSES, "img_size": IMG, "arch": "densenet121"},
           OUT_DIR / "condition_classifier.pt")
print("saved", OUT_DIR / "condition_classifier.pt")

## Experiment log (fill in one row per run; this is the evidence for "systematic and iterative" optimisation)

| Run | What changed | Why | Val macro-F1 | Caries recall | Notes |
|---|---|---|---|---|---|
| 1 | Baseline: EfficientNet-B0, 224 px, class-weighted loss, 5 frozen epochs (lr 1e-3) then fine-tune (lr 1e-4), early stop patience 5 | starting point | 0.971 (fine-tune epoch 12; stopped at 17) | 0.840 (test, 21/25) | Kaggle version 1, GPU T4. Test acc 0.979, macro-F1 0.964. 58 images in 29 near-duplicate groups at DUP_SIM 0.95, groups checked by eye: real mirrored/re-cropped copies, 0 label conflicts. Train F1 ~0.99 vs val ~0.96: mild overfitting. Grad-CAM: Calculus heat on photo edge, Hypodontia on rotated black border = possible shortcut |
| 2 | Backbone EfficientNet-B0 -> DenseNet-121, AND data Kaggle oral_clean -> Roboflow oral-diseases (6 classes: adds Gingivitis, no Normal) | a teammate trains EfficientNet on the Kaggle set; more photos | | | two things changed at once, so this is not a like-for-like comparison with run 1 |